In [ ]:
# Load our libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from tensorflow import keras
from tensorflow.keras.callbacks import EarlyStopping
import seaborn as sns
import matplotlib.pyplot as plt

## 1. Formulate / Outline the problem

We'll now shift focus to another dataset and another problem. Weather prediction, in specific, **predict sunshine hours for Basel**.

Sunshine hours is a numeric variable, and therefore our problem is no longer classification (into categories) but rather **regression** (for numbers / continuous values).

## 2. Identify inputs and outputs

In [ ]:
# Load the data
data = pd.read_csv("https://zenodo.org/record/5071376/files/weather_prediction_dataset_light.csv?download=1")

In [ ]:
# Print some info for our data
print("Number of (rows, columns): ", data.shape)
print("Column names: ", data.columns)

In [ ]:
# Look at first five rows
data.head()

In [ ]:
# See what data types we have and if the dataset is complete
data.info()

Some things we can conclude from our dataset:
- 91 columns: date (yyymmdd format), month (numeric), and weather variables for 11 different cities in Europe.
- 3654 rows, meaning 3654 days of collected data (or 10 years).
- All variables/features are numeric
- It doesn't seem like we have empty/null values
- We want to predict `BASEL_sunshine` on day `i+1` based on all other variables on day `i`.

## 3. Prepare data

As usual, our data preparation requires: data cleaning, defining output/target/y and inputs/features/x, and splitting intro train and test sets

In [ ]:
## First, build the models as we only had information for 3 years
nr_rows = 365*3 # 3 years

# Prepare features. We won't be using DATE or MONTH
X_data = data.loc[0:nr_rows]
X_data = X_data.drop(columns=['DATE', 'MONTH'])

# Prepare target, knowing that our target is BASEL_sunshine for day i+1
y_data = data.loc[1:(nr_rows + 1), "BASEL_sunshine"]

### Data Splitting Strategy: Train, Validation, and Test

To build a model that actually works in the real world, we are moving from a 2-way split to a **3-way partition**.

Why the change?
Previously, we used the `test` set for evaluation. However, every time we "tweak" the model to get a better score on that test set, we are accidentally leaking information. The model is no longer being tested on "unseen" data because the developer has shaped the model to fit that specific data.

The Three Partitions

* **Train Set**: The "Textbook." This is the data the model looks at to learn patterns and adjust its weights.
* **Validation Set**: The "Practice Exam." We use this during training to compare different architectures and tune hyperparameters. The model doesn't learn from this data directly, but we use its results to decide which model version is best.
* **Test Set**: The "Final Exam." This is a locked vault. It is used **only once** at the very end to provide an unbiased measure of how the final model will perform in the real world.

In short:
| Split | Purpose | Used for Training? | Influences Model Design? |
| :--- | :--- | :--- | :--- |
| **Train** | Learning weights | Yes | Yes |
| **Validation** | Tuning & Selection | No | Indirectly (via the developer) |
| **Test** | Final Evaluation | No | No |

In [ ]:
# Split intro train and holdout (which will later be split into validation and test)
# A conventional 70% of the data for training
X_train, X_holdout, y_train, y_holdout = train_test_split(X_data, y_data, test_size=0.3, random_state=0)

# Using our holdout, split evenly (15/15%) into validation and test
X_val, X_test, y_val, y_test = train_test_split(X_holdout, y_holdout, test_size=0.5, random_state=0)